In [ ]:
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore
import statsmodels.stats.multitest as smm
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt


In [ ]:
# 从Excel文件读取数据
def read_data_from_excel(t_values_file, gene_expression_file):
    """从Excel文件读取t_values和gene_expression数据"""
    t_values = pd.read_csv(t_values_file, header=None)
    gene_expression_df = pd.read_csv(gene_expression_file)
    # HC_common = pd.read_excel(HC_common)
    # MDDNSI_common = pd.read_excel(MDDNSI_common).values

    # HC_MDDNSI_diff = HC_common - MDDNSI_common

    # Step 1: Find coordinates in df1 where values are < 0.05
    coordinates = [(col_idx, row_idx) for row_idx in range(t_values.shape[0]) for col_idx in range(row_idx + 1, t_values.shape[1]) if t_values.iat[row_idx, col_idx] != 0]

    # Step 2: Format coordinates as "col_index_row_index" and create DataFrame
    formatted_coordinates = [f"{row}-{col}" for col, row in coordinates]
    # df_coordinates = pd.DataFrame({"Coordinate": formatted_coordinates})
    # Step 3: Filter df2 based on matching coordinates in the "Region_Pair" column
    df_filtered = gene_expression_df[gene_expression_df["Region_Pair"].isin(formatted_coordinates)].reset_index(drop=True)
    df_filtered = df_filtered.drop(columns=['Region_Pair'])
     # Step 4:
    df_diff = pd.DataFrame([(f"{row}-{col}", t_values.at[row, col]) for col, row in coordinates], columns=['Region_Pair', 'Value'])

    diff_values = df_diff.drop(columns=['Region_Pair'])
    
    gene_names = gene_expression_df.columns.values  # 获取基因名字作为列名

    return df_diff, diff_values.values, df_filtered.values, gene_names

# K-fold 交叉验证选择最佳成分数
def optimal_pls_components(X, Y, max_components=15, n_splits=5):
    """通过K-fold交叉验证选择最佳PLS成分数"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_errors = []

    for n_components in range(1, max_components + 1):
        fold_errors = []
        for train_index, test_index in kf.split(X):
            X_train, X_test = X[train_index], X[test_index]
            Y_train, Y_test = Y[train_index], Y[test_index]
            
            pls = PLSRegression(n_components=n_components)
            pls.fit(X_train, Y_train)
            Y_pred = pls.predict(X_test)
            
            # 计算均方根误差 (RMSE)
            rmse = np.sqrt(np.mean((Y_test - Y_pred) ** 2))
            fold_errors.append(rmse)

        cv_errors.append(np.mean(fold_errors))

    return cv_errors

# 置换检验函数
def permutation_test_weights(X, Y, best_n_components, n_permutations=1000):
    pls = PLSRegression(n_components=best_n_components)
    pls.fit(X, Y)
    original_weights = pls.x_weights_

    explained_variance = np.var(pls.x_scores_, axis=0)
    explained_variance_ratio = explained_variance / np.sum(explained_variance)
    
    # 进行置换检验
    permuted_variances = np.zeros((n_permutations, best_n_components))
    for i in range(n_permutations):
        Y_permuted = np.random.permutation(Y)
        pls.fit(X, Y_permuted)
        permuted_variances[i, :] = np.var(pls.x_scores_, axis=0)

    p_values = np.mean(permuted_variances >= explained_variance, axis=0)
    
    return explained_variance_ratio, p_values, original_weights

# 基于显著性结果选择最佳成分和基因
def select_best_component_and_genes(explained_variance_ratio, p_values, weights, threshold=0.05):
    # 优先选择解释方差最大的显著成分
    significant_components = np.where(p_values < threshold)[0]
    if len(significant_components) > 0:
        best_component = significant_components[np.argmax(explained_variance_ratio[significant_components])]
    else:
        best_component = np.argmax(explained_variance_ratio)

    # 选择显著成分中权重显著的基因
    best_weights = weights[:, best_component]
    z_scores = zscore(best_weights)
    
    # 计算每个基因的p值
    p_values_genes = np.array([np.mean(np.random.permutation(z_scores) >= abs(z)) for z in z_scores])
    # 选择p值小于0.05的基因
    best_genes = np.where(p_values_genes < 0.05)[0]
    # # 确保p_values_genes是一维数组
    # assert p_values_genes.ndim == 1, "p_values_genes must be a 1D array."
    
    # # 使用Benjamini-Hochberg FDR校正进行多重比较校正
    # rejected, p_values_corrected = smm.fdrcorrection(p_values_genes, alpha=threshold, method='negcorr')

    # # 选择经过校正后显著的基因
    # best_genes = np.where(rejected)[0]

    return best_component, best_genes, z_scores, p_values_genes


# 保存显著基因结果到CSV文件
def save_best_genes_to_csv(gene_names, best_genes, z_scores, p_values_genes, output_file):
    """将显著基因的相关信息保存到CSV文件中"""
    results = {
        'Gene Index': gene_names[best_genes],
        'Original Z-Score': z_scores[best_genes],
        'P-Value': p_values_genes[best_genes]
    }
    results_df = pd.DataFrame(results)
    results_df.sort_values(by='P-Value', inplace=True)

    results_df.to_csv(output_file, index=False)

# 绘图函数
def plot_rmse_and_variance(cv_errors, explained_variance_ratio,fig_outputfile):
    """绘制RMSE和可解释方差比例的折线图"""
    components_cv_errors = np.arange(1, len(cv_errors) + 1)
    components_explained = np.arange(1, len(explained_variance_ratio) + 1)

    plt.figure(figsize=(12, 6))

    # 绘制RMSE
    plt.subplot(1, 2, 1)
    plt.plot(components_cv_errors, cv_errors, marker='o', color='blue')
    plt.title('RMSE vs Number of PLS Components')
    plt.xlabel('Number of PLS Components')
    plt.ylabel('RMSE')
    plt.xticks(components_cv_errors)
    plt.yticks(np.arange(0,2,0.1))

    # 绘制可解释方差比例
    plt.subplot(1, 2, 2)
    plt.plot(components_explained, explained_variance_ratio, marker='o', color='orange')
    plt.title('Explained Variance Ratio vs Number of PLS Components')
    plt.xlabel('Number of PLS Components')
    plt.ylabel('Explained Variance Ratio')
    plt.xticks(components_explained)
    plt.yticks(np.arange(0,1.2,0.1))

    plt.tight_layout()
    plt.savefig(fig_outputfile, dpi=300)
    plt.show()

In [ ]:
# 从Excel文件读取数据
gene_expression_file = r"E:\aliyun_backup\muilt_disorders\13_Allen\gene_contribution_df.csv"  

In [ ]:
t_values_file = r"E:\aliyun_backup\muilt_disorders\11_pls\result\pls_weight\偏离连接_SCZ_top10.csv"

In [ ]:
t_values_df, t_values, gene_expression, gene_names = read_data_from_excel(t_values_file, gene_expression_file)
t_values_df.to_csv(r'E:\aliyun_backup\muilt_disorders\13_Allen\偏离连接_SCZ_t_values.csv', index=False)

In [ ]:
# 标准化数据
scaler = StandardScaler()
X = scaler.fit_transform(gene_expression)
Y = scaler.fit_transform(t_values)

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
# 选择最佳PLS成分数
cv_errors = optimal_pls_components(X, Y)

# 选择最佳成分数
best_n_components = np.argmin(cv_errors) + 1  # +1 因为索引从0开始
print(f"Optimal number of PLS components: {best_n_components}")

In [ ]:
# 计算成分的解释方差和显著性
explained_variance_ratio, p_values, weights = permutation_test_weights(X, Y, best_n_components)

# 选择最佳成分和显著基因
best_component, best_genes, z_scores, p_values_genes = select_best_component_and_genes(explained_variance_ratio, p_values, weights)


In [ ]:
p_values

In [ ]:
print(f"Best component: {best_component + 1}")
print(f"Genes significantly associated with t-values: {best_genes}")

In [ ]:
output_file = r'E:\aliyun_backup\muilt_disorders\13_Allen\best_genes_SCZ.csv'  # 替换为输出文件路径
# 保存显著基因结果到CSV文件
save_best_genes_to_csv(gene_names, best_genes, z_scores, p_values_genes, output_file)


In [ ]:
# 绘制RMSE和可解释方差比例的折线图
fig_outputfile = r'E:\aliyun_backup\muilt_disorders\13_Allen\rmse_and_variance_偏离连接_SCZ.tif'
plot_rmse_and_variance(cv_errors, explained_variance_ratio,fig_outputfile)